In [19]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [22]:
# Results generated by our library
csv_our = pd.read_csv("results_our/results_our.csv")

# Results generated by the reference library
csv_ref = pd.read_csv("results_ref/results_ref.csv")

In [23]:
keys = ["dgp", "statistic", "method", "n", "B", "repetitions"]

# Merge the datasets so we can analyze the same cases together
comparison = csv_our.merge(
    csv_ref,
    on=keys,
    suffixes=("_our", "_ref"),
    validate="one_to_one",
)

assert len(comparison) == len(csv_ref) and len(comparison) == len(csv_our), (
    "Experiments don't match!"
)

comparison.head(5)

,method,nominal_1_sided_coverage_our,nominal_2_sided_coverage_our,lower_coverage_conditional_our,upper_coverage_conditional_our,two_sided_coverage_conditional_our,two_sided_coverage_our,mean_two_sided_length_conditional_our,median_two_sided_length_conditional_our,lower_bound_success_rate_our,...,nominal_2_sided_coverage_ref,lower_coverage_conditional_ref,upper_coverage_conditional_ref,two_sided_coverage_conditional_ref,two_sided_coverage_ref,mean_two_sided_length_conditional_ref,median_two_sided_length_conditional_ref,lower_bound_success_rate_ref,upper_bound_success_rate_ref,two_sided_success_rate_ref
0,percentile,0.975,0.95,0.941,0.939,0.880,0.880,1.243768,1.218740,1.0,...,0.95,0.944,0.936,0.880,0.880,1.242883,1.220705,1.0,1.0,1.0
1,double,0.975,0.95,0.967,0.964,0.931,0.931,1.686106,1.655035,1.0,...,0.95,0.967,0.964,0.931,0.931,1.692402,1.666504,1.0,1.0,1.0
2,percentile,0.975,0.95,0.964,0.964,0.928,0.928,0.677151,0.674429,1.0,...,0.95,0.963,0.966,0.929,0.929,0.678435,0.678903,1.0,1.0,1.0
3,double,0.975,0.95,0.972,0.974,0.946,0.946,0.724669,0.725210,1.0,...,0.95,0.974,0.973,0.947,0.947,0.723186,0.718664,1.0,1.0,1.0
4,percentile,0.975,0.95,0.970,0.972,0.942,0.942,0.345786,0.345058,1.0,...,0.95,0.969,0.971,0.940,0.940,0.345475,0.344736,1.0,1.0,1.0


In [29]:
bounds = ["lower_bound", "upper_bound", "two_sided"]

# None of the experiments lead to construction of invalid intervals for our library
for bound in bounds:
    assert comparison[f"{bound}_success_rate_our"].eq(1).all()

In [48]:
# Computation of Pearson correlation can lead to division by 0 error for small sample sizes
# -> failure to construct a valid CI in 3/1000 simulations for the bootstrap_ci library
# (for both percentile and double methods)
success_cols = [f"{bound}_success_rate_ref" for bound in bounds]

failure_cases_ref = comparison.loc[
    comparison[success_cols].ne(1).any(axis=1),
    ["dgp", "method", "n", *success_cols],
]

failure_cases_ref

,dgp,method,n,lower_bound_success_rate_ref,upper_bound_success_rate_ref,two_sided_success_rate_ref
24,DGPBiNorm-1_1_2.0_0.5_1.0,percentile,8,0.997,0.997,0.997
25,DGPBiNorm-1_1_2.0_0.5_1.0,double,8,0.997,0.997,0.997


In [ ]:
# Compute unconditional coverage from the conditional coverage and analyze the
# difference between the results.
# Instead of using (#covers / #successful_intervals), use (#covers / #all_repetitions)
# by multiplying the coverage by the success_rate, effectively treating
# unsuccessful constructions as non-coverage (in our case, this will have
# a very small effect on the reference library coverage for the bivariate normal
# case with n = 8)
coverage_columns = ["lower_coverage", "upper_coverage", "two_sided_coverage"]
success_rate_columns = [
    "lower_bound_success_rate",
    "upper_bound_success_rate",
    "two_sided_success_rate",
]

for cov_col, rate_col in zip(coverage_columns, success_rate_columns):
    comparison[f"{cov_col}_unconditional_our"] = (
        comparison[f"{cov_col}_conditional_our"] * comparison[f"{rate_col}_our"]
    )
    comparison[f"{cov_col}_unconditional_ref"] = (
        comparison[f"{cov_col}_conditional_ref"] * comparison[f"{rate_col}_ref"]
    )

    comparison[f"{cov_col}_unconditional_difference"] = (
        comparison[f"{cov_col}_unconditional_our"]
        - comparison[f"{cov_col}_unconditional_ref"]
    )

coverage_summary = pd.DataFrame(
    {
        column: {
            "mean_difference": comparison[
                f"{column}_unconditional_difference"
            ].mean(),
            "mean_absolute_difference": comparison[
                f"{column}_unconditional_difference"
            ]
            .abs()
            .mean(),
            "maximum_absolute_difference": comparison[
                f"{column}_unconditional_difference"
            ]
            .abs()
            .max(),
        }
        for column in coverage_columns
    }
)
coverage_summary


,lower_coverage,upper_coverage,two_sided_coverage
mean_difference,-0.000071,0.001333,0.001119
mean_absolute_difference,0.002262,0.002381,0.002833
maximum_absolute_difference,0.008000,0.008000,0.010000
